# Recursive CTE

# 1. Load packages

In [1]:
import os
from dotenv import load_dotenv
import mysql.connector
import pandas as pd

# 2. Connect to MySQL Server

In [2]:
# Load .env
# load_dotenv()

# Force load
load_dotenv(dotenv_path=".env")   # force load

conn = mysql.connector.connect(
    user = os.getenv("MYSQL_USER"),
    host = os.getenv("MYSQL_HOST"),
    port = os.getenv("MYSQL_PORT"),
    password = os.getenv("MYSQL_PASSWORD"),
    database = "recursive_cte_db"
)
print("Connected to the MySQL server successfully")

Connected to the MySQL server successfully


# 3. Create a Cursor

In [3]:
cursor = conn.cursor()

# 4. Run SQL Query

In [4]:
cursor.execute("SELECT * FROM employeesTbl")

# 5. Fetch query results 

In [5]:
employees = cursor.fetchall()

## 5.1 Print the results as a list of tuples

Each tuple corresponds to a record or row in the SQL table.

In [6]:
employees

[(1, 'Tom', 2),
 (2, 'Josh', None),
 (3, 'Mike', 2),
 (4, 'John', 3),
 (5, 'Pam', 1),
 (6, 'Mary', 3),
 (7, 'James', 1),
 (8, 'Sam', 5),
 (9, 'Simon', 1)]

## 5.2 Print the results as a DataFrame (without headers)

In [7]:
pd.DataFrame(employees)

,0,1,2
0,1,Tom,2.0
1,2,Josh,NaN
2,3,Mike,2.0
3,4,John,3.0
4,5,Pam,1.0
5,6,Mary,3.0
6,7,James,1.0
7,8,Sam,5.0
8,9,Simon,1.0


## 5.3 Print the results as a DataFrame (with headers)

In [8]:
cursor.description

[('employee_id', 3, None, None, None, None, 0, 53251, 63),
 ('name', 253, None, None, None, None, 1, 0, 255),
 ('manager_id', 3, None, None, None, None, 1, 32768, 63)]

In [9]:
headers = [col[0] for col in cursor.description]
headers

['employee_id', 'name', 'manager_id']

In [10]:
df = pd.DataFrame(employees, columns=headers)
df

,employee_id,name,manager_id
0,1,Tom,2.0
1,2,Josh,NaN
2,3,Mike,2.0
3,4,John,3.0
4,5,Pam,1.0
5,6,Mary,3.0
6,7,James,1.0
7,8,Sam,5.0
8,9,Simon,1.0


## 5.4 Change the type of `manager_id` column to `Int64`

In [11]:
df["manager_id"] = df["manager_id"].astype("Int64")

"""
df = df.astype({
    "employee_id": "Int64",
    "manager_id": "Int64"
})
"""

df

,employee_id,name,manager_id
0,1,Tom,2
1,2,Josh,<NA>
2,3,Mike,2
3,4,John,3
4,5,Pam,1
5,6,Mary,3
6,7,James,1
7,8,Sam,5
8,9,Simon,1


# 6. Employee-Manager Relationships

In [12]:
# MySQL self-join query
query = """
SELECT * FROM
employeesTbl E
LEFT JOIN
employeesTbl M
ON E.manager_id = M.employee_id
"""

# Execute the query
cursor.execute (query)

# Print the results of the query
results = cursor.fetchall()
headers = [col[0] for col in cursor.description]
display(pd.DataFrame(results, columns=headers))

,employee_id,name,manager_id,employee_id,name,manager_id
0,1,Tom,2.0,2.0,Josh,NaN
1,2,Josh,NaN,NaN,None,NaN
2,3,Mike,2.0,2.0,Josh,NaN
3,4,John,3.0,3.0,Mike,2.0
4,5,Pam,1.0,1.0,Tom,2.0
5,6,Mary,3.0,3.0,Mike,2.0
6,7,James,1.0,1.0,Tom,2.0
7,8,Sam,5.0,5.0,Pam,1.0
8,9,Simon,1.0,1.0,Tom,2.0


In [13]:
# MySQL self-join query for getting employee-manager relations
query = """
SELECT E.name AS employee, M.name AS manager FROM
employeesTbl E
LEFT JOIN
employeesTbl M
ON E.manager_id = M.employee_id
"""

cursor.execute(query)
results = cursor.fetchall()
headers = [col[0] for col in cursor.description]

pd.DataFrame(results, columns=headers)

,employee,manager
0,Tom,Josh
1,Josh,None
2,Mike,Josh
3,John,Mike
4,Pam,Tom
5,Mary,Mike
6,James,Tom
7,Sam,Pam
8,Simon,Tom


# 7. Recursive CTE : Levels of Employees in Organizational Hierarchy

In [14]:
query = """
SELECT E.employee_id, E.name AS employee, M.name AS manager, 1 AS level FROM 
employeesTbl E
LEFT JOIN
employeesTbl M
ON E.manager_id = M.employee_id
WHERE E.manager_id IS NULL
"""
cursor.execute(query)
results = cursor.fetchall()
headers = [col[0] for col in cursor.description]
pd.DataFrame(results, columns=headers)

,employee_id,employee,manager,level
0,2,Josh,None,1


In [15]:
query = """
WITH RECURSIVE rec_cte AS (
SELECT E.employee_id, E.name AS employee, M.name AS manager, 1 AS level FROM 
employeesTbl E
LEFT JOIN
employeesTbl M
ON E.manager_id = M.employee_id
WHERE E.manager_id IS NULL

UNION ALL

SELECT E.employee_id, E.name AS employee, R.employee AS manager, R.level+1 AS level FROM
rec_cte R
JOIN
employeesTbl E
ON R.employee_id = E.manager_id
) SELECT * FROM rec_cte
"""
cursor.execute(query)
results = cursor.fetchall()
headers = [col[0] for col in cursor.description]
pd.DataFrame(results, columns=headers)

,employee_id,employee,manager,level
0,2,Josh,None,1
1,1,Tom,Josh,2
2,3,Mike,Josh,2
3,4,John,Mike,3
4,5,Pam,Tom,3
5,6,Mary,Mike,3
6,7,James,Tom,3
7,9,Simon,Tom,3
8,8,Sam,Pam,4


In [16]:
query = """
WITH RECURSIVE rec_cte AS (
SELECT E.employee_id, E.name AS employee, M.name AS manager, 1 AS level FROM 
employeesTbl E
LEFT JOIN
employeesTbl M
ON E.manager_id = M.employee_id
WHERE E.manager_id IS NULL

UNION ALL

SELECT E.employee_id, E.name AS employee, R.employee AS manager, R.level+1 AS level FROM
rec_cte R
JOIN
employeesTbl E
ON R.employee_id = E.manager_id
) SELECT employee, IFNULL(manager, 'HOD') AS manager, level FROM rec_cte
"""
cursor.execute(query)
results = cursor.fetchall()
headers = [col[0] for col in cursor.description]
pd.DataFrame(results, columns=headers)

,employee,manager,level
0,Josh,HOD,1
1,Tom,Josh,2
2,Mike,Josh,2
3,John,Mike,3
4,Pam,Tom,3
5,Mary,Mike,3
6,James,Tom,3
7,Simon,Tom,3
8,Sam,Pam,4


# 8. Recursive CTE : Organizational Hierarchy

In [17]:
cursor.execute("SET @N = 8")

cursor.execute("""
WITH RECURSIVE rec_cte AS (
SELECT E.name AS employee, M.name AS manager, M.manager_id AS manager_id_of_manager FROM
employeesTbl E
LEFT JOIN
employeesTbl M
ON E.manager_id = M.employee_id
WHERE E.employee_id = @N

UNION ALL

SELECT R.manager AS employee, E.name AS manager, E.manager_id AS manager_id_of_manager FROM
rec_cte R
JOIN
employeesTbl E
ON R.manager_id_of_manager = E.employee_id
) SELECT * FROM rec_cte
""")

results = cursor.fetchall()
headers = [col[0] for col in cursor.description]
pd.DataFrame(results, columns=headers)

,employee,manager,manager_id_of_manager
0,Sam,Pam,1.0
1,Pam,Tom,2.0
2,Tom,Josh,NaN


In [18]:
cursor.execute("SET @N = 8")

cursor.execute("""
WITH RECURSIVE rec_cte AS (
SELECT E.name AS employee, M.name AS manager, M.manager_id AS manager_id_of_manager FROM
employeesTbl E
LEFT JOIN
employeesTbl M
ON E.manager_id = M.employee_id
WHERE E.employee_id = @N

UNION ALL

SELECT R.manager AS employee, E.name AS manager, E.manager_id AS manager_id_of_manager FROM
rec_cte R
JOIN
employeesTbl E
ON R.manager_id_of_manager = E.employee_id
) SELECT employee, manager FROM rec_cte
""")

results = cursor.fetchall()
headers = [col[0] for col in cursor.description]
pd.DataFrame(results, columns=headers)

,employee,manager
0,Sam,Pam
1,Pam,Tom
2,Tom,Josh


# 9. Stored Procedure

In [19]:
query = """
CREATE PROCEDURE IF NOT EXISTS org_hier(IN N INT)
BEGIN
WITH RECURSIVE rec_cte AS (
SELECT E.name AS employee, M.name AS manager, M.manager_id AS manager_id_of_manager FROM
employeesTbl E
LEFT JOIN
employeesTbl M
ON E.manager_id = M.employee_id
WHERE E.employee_id = N

UNION ALL

SELECT R.manager AS employee, E.name AS manager, E.manager_id AS manager_id_of_manager FROM
rec_cte R
JOIN
employeesTbl E
ON R.manager_id_of_manager = E.employee_id
) SELECT employee, manager FROM rec_cte;
END
"""
cursor.execute(query)

In [20]:
cursor.callproc("org_hier", (8,))

for result in cursor.stored_results():
    rows = result.fetchall()
    headers = [col[0] for col in result.description]
    
    df = pd.DataFrame(rows, columns=headers)
    display(df)

C:\Users\adeet\AppData\Local\Temp\ipykernel_7488\4059545724.py:3: DeprecationWarning: Call to deprecated function stored_results. Reason: The property counterpart 'stored_results' will be added in a future release, and this method will be removed.
  for result in cursor.stored_results():


,employee,manager
0,Sam,Pam
1,Pam,Tom
2,Tom,Josh
